In [10]:
!pip -q install transformers

In [15]:
import re

text = """
DOCUMENT 1
Title: Artificial Intelligence
Artificial intelligence enables machines to perform intelligent tasks.
It is widely used in modern applications.

----------------------------------------

DOCUMENT 2
Title: Natural Language Processing
NLP helps computers understand human language.
It is used in translation and text analysis.

----------------------------------------

DOCUMENT 3
Title: Machine Learning
Machine learning allows systems to learn from data.
It is an important field of artificial intelligence.
"""

lines = [line.strip() for line in text.splitlines()]

def heading_probability(line):
    if not line:
        return 0.0

    score = 0.0

    if re.match(r"^DOCUMENT\s+\d+", line, re.I):
        score += 0.75

    if re.match(r"^TITLE\s*:", line, re.I):
        score += 0.60

    words = line.split()
    if len(words) <= 6 and line[0].isupper():
        score += 0.15

    return min(score, 1.0)

def detect_boundaries(lines, threshold=0.60):
    documents = []
    current_document = []
    state = "OUTSIDE"

    for line in lines:
        probability = heading_probability(line)

        if probability >= threshold:
            if current_document:
                documents.append(current_document)

            current_document = [line]
            state = "INSIDE"

        elif re.fullmatch(r"[-=_]{5,}", line):
            if state == "INSIDE" and current_document:
                documents.append(current_document)
                current_document = []

            state = "OUTSIDE"

        elif line:
            if state == "OUTSIDE":
                current_document = []
                state = "INSIDE"

            current_document.append(line)

    if current_document:
        documents.append(current_document)

    return documents

documents = detect_boundaries(lines)

for i, document in enumerate(documents, start=1):
    print(f"\n--- Detected Document {i} ---")
    print("\n".join(document))


--- Detected Document 1 ---
DOCUMENT 1

--- Detected Document 2 ---
Title: Artificial Intelligence
Artificial intelligence enables machines to perform intelligent tasks.
It is widely used in modern applications.

--- Detected Document 3 ---
DOCUMENT 2

--- Detected Document 4 ---
Title: Natural Language Processing
NLP helps computers understand human language.
It is used in translation and text analysis.

--- Detected Document 5 ---
DOCUMENT 3

--- Detected Document 6 ---
Title: Machine Learning
Machine learning allows systems to learn from data.
It is an important field of artificial intelligence.
